# ABI-S15 Write Up Attempt

## Data & Embedding - Foundations

To explore whether conversational convergence and divergence can be compuationally detected in multimodal team conversations, I began with ABI_S15 as a pilot case. This session was selected because it contains multiple decision points and substantive technical discussion, making it a strong trestbed for validating whether automated methods can capture meaningful group dynamics. The transcript was segmented into sentences-level units, which served as the smallest analytic granularity. Each row of the resulting dataset contained the text of the sentence, its speaker identifier, and a timestamp (in seconds). Codebook annotations - such as *decision*, *explanation*, and *new idea* - were aligned at the sentence or phrase level, allowing me to later compare computationally detected convergence with human-coded markers.

Each sentence $s_i$ was then mapped into a dense vector embedding using a pretrained transformer model (SentenceTransformer, `all-MiniLM-L6-v2`). Formally, the transcript is represented as a sequence

$D = \{(s_i, t_i, u_i)\}^{N}_{i=1}$

Where $s_i$ is the sentencce text, $t_i$ the timestamp, and $u_i$ the speaker. The embedding function $f(\cdot)$ maps each sentence into a $d$-dimensional semantic space:

$e_i = f(s_i), e_i \in \mathbb{R}^d$

In practice, $d = 384$ for MiniLM. For interpretabiilty and numerical stability, all embeddings were $L_2$-normalized, giving unit vectors $\tilde{e_i}$.

Semantic similiarity between any two sentences is then defined by cosine similarity:

$sim(\tilde{e_i}, \tilde{e_j} = \tilde{e_i}^T\tilde{e_j})$

which reduces to the dot product of the normalized embeddings. Scores closer to 1 indicate high semantic overlap, while lower scores reflect divergence. This embedding space establishes the foundation for detecting moments when participants independently produce semantically similar contributions in close temporal proximity - our operational definition of conservational convergence

## Convergence Detection via Time-Bounded Bursts

Having established a semantic similarity measure at the sentence level, the next step is to detect *bursts of convergence* - intervals where multiple  speakers independently articulate semantically similar content within a short period of time.

To operationalize this, I defined **temporal windows** centered at either *(a)* fixed increments (sliding windows) or *(b)* pre-coded "decision-like" lines. For each window $W_k$, defined by a center $\tau_k$ and a half-width $\Delta$, the set of sentences included is:

$W_k = \{i | t_i - \tau_k \leq \Delta\}$

Where $t_i$ denotes the timestamp of sentence $i$. In this analysis, $\Delta = 45$ seconds.

Within each window, I considered all cross-speaker pairs of sentences that occur within a smaller temporal gap $\delta$, in order to capture near-simultaneous contributions rather than statement far apart in time. The set of candidate pairs is:

$\mathcal{P}_k = \{(i,j) \in W_k \times W_k | i< j, | t_i - t_j | \leq \delta, u_i \neq u_j\},$

where $u_i$ denotes the speaker identity.

For each window, I then computed a **burst score** that quantifies the proportion of cross-speakers that exceeded a similarity threshold $\theta$:

$B(W_k) = \frac{1}{|\mathcal{P}_k|}\sum_{(i,j) \in \mathcal{P}_k}1[\tilde{e_i}^T\tilde{e_j} \geq \theta],$

where $1[\cdot]$ is the indicator function. Intuitively, $B(W_k)$ captures the density of semantically overlapping contributions in the window.

A window $W_k is flagged as a **convergence burst** if two conditions are met:

1. The burst score exceeds a threshold $(B(W_k) \geq \beta)$ and 

2. Contributions come from at least $m$ distinct speakers

For ABI_S15, thresholds were selected empircally $(\theta \approx 0.75, \beta \approx 0.20, m = 3)$. Flagged windwos (e.g. the 913-918s segment) represent candidate convergence moments, which can then be validated against transcript content and codebook labels.